In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from imblearn.over_sampling import SMOTE
import xgboost as xgb


In [2]:
data=pd.read_csv("Datasets/TRAIN_DATA.csv")
test_data=pd.read_csv("Datasets/TEST_DATA.csv")


In [3]:
print(len(data))
print(len(test_data))


630000
70000


In [4]:
data.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,55,2,89,4.7,7.6,6.8,30.7,0.91,131,...,Female,White,Graduate,Low,Never,Employed,0,0,0,1.0
1,1,50,1,59,7.1,4.9,8.6,27.1,0.87,113,...,Male,White,Graduate,Middle,Current,Employed,0,0,0,1.0
2,2,34,4,46,1.7,6.6,4.0,32.0,0.94,119,...,Female,White,Highschool,Lower-Middle,Current,Unemployed,0,0,0,0.0
3,3,69,2,245,3.9,6.0,3.5,30.1,0.92,135,...,Male,Black,Graduate,Lower-Middle,Never,Employed,0,0,0,0.0
4,4,44,2,62,6.4,7.9,5.6,22.5,0.83,104,...,Female,White,Graduate,Lower-Middle,Never,Employed,0,0,0,0.0


In [5]:
data.info()         
data.describe()  

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 630000 entries, 0 to 629999
Data columns (total 26 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  630000 non-null  int64  
 1   age                                 630000 non-null  int64  
 2   alcohol_consumption_per_week        630000 non-null  int64  
 3   physical_activity_minutes_per_week  630000 non-null  int64  
 4   diet_score                          630000 non-null  float64
 5   sleep_hours_per_day                 630000 non-null  float64
 6   screen_time_hours_per_day           630000 non-null  float64
 7   bmi                                 630000 non-null  float64
 8   waist_to_hip_ratio                  630000 non-null  float64
 9   systolic_bp                         630000 non-null  int64  
 10  diastolic_bp                        630000 non-null  int64  
 11  heart_rate                

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,diastolic_bp,heart_rate,cholesterol_total,hdl_cholesterol,ldl_cholesterol,triglycerides,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
count,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000,630000.000000
mean,314999.500000,50.363227,2.071648,80.231557,5.963251,7.001907,6.012962,25.874561,0.858755,116.295073,75.440187,70.171635,186.826567,53.823516,102.914338,123.084843,0.149430,0.181995,0.030349,0.623295
std,181865.479132,11.652836,1.047572,51.201002,1.463775,0.901905,2.022679,2.860022,0.037975,11.008478,6.824568,6.938096,16.734801,8.269259,19.024615,24.749024,0.356512,0.385841,0.171546,0.484560
min,0.000000,19.000000,1.000000,1.000000,0.100000,3.100000,0.600000,15.100000,0.680000,91.000000,51.000000,42.000000,117.000000,21.000000,51.000000,31.000000,0.000000,0.000000,0.000000,0.000000
25%,157499.750000,42.000000,1.000000,49.000000,5.000000,6.400000,4.600000,23.900000,0.830000,108.000000,71.000000,65.000000,175.000000,48.000000,89.000000,106.000000,0.000000,0.000000,0.000000,0.000000
50%,314999.500000,50.000000,2.000000,71.000000,6.000000,7.000000,6.000000,25.900000,0.860000,116.000000,75.000000,70.000000,187.000000,54.000000,103.000000,123.000000,0.000000,0.000000,0.000000,1.000000
75%,472499.250000,58.000000,3.000000,96.000000,7.000000,7.600000,7.400000,27.800000,0.880000,124.000000,80.000000,75.000000,199.000000,59.000000,116.000000,139.000000,0.000000,0.000000,0.000000,1.000000
max,629999.000000,89.000000,9.000000,747.000000,9.900000,9.900000,16.500000,38.400000,1.050000,163.000000,104.000000,101.000000,289.000000,90.000000,205.000000,290.000000,1.000000,1.000000,1.000000,1.000000


In [6]:
data.isnull().sum()  
data.duplicated().sum()

np.int64(0)

# Drop id cause it's unique

In [7]:
datasample = data.drop(columns=['id'])
datasample.head()

test_data = test_data.drop(columns=['id'], errors='ignore')

# data visualization

# split data


In [8]:
# STEP 1: Prepare the data
X = datasample.drop('diagnosed_diabetes', axis=1)
y = datasample['diagnosed_diabetes'].astype(int)  # Convert to integer here

print("Target dtype:", y.dtype)
print("Class balance:", y.value_counts(normalize=True).round(3))

print("\n" + "="*60)
print("STEP 2: CREATE VALIDATION SPLIT")
print("="*60)

from sklearn.model_selection import train_test_split


X_train_main, X_val, y_train_main, y_val = train_test_split(
    X, y, 
    test_size=0.2,  
    random_state=42,
    stratify=y  # NOT y_train, use y
)

print(f"Main training: {X_train_main.shape}")
print(f"Validation:     {X_val.shape}")

# Check if test_data exists
if 'test_data' in locals():
    print(f"Teacher's test: {test_data.shape}")
else:
    print("Teacher's test: Not loaded yet")

Target dtype: int64
Class balance: diagnosed_diabetes
1    0.623
0    0.377
Name: proportion, dtype: float64

STEP 2: CREATE VALIDATION SPLIT
Main training: (504000, 24)
Validation:     (126000, 24)
Teacher's test: (70000, 24)


In [9]:
print("🔍 CHECKING YOUR DATA:")
print("="*60)

# Check what variables you ACTUALLY have
print("Variables you have created:")
if 'X_train_main' in locals():
    print(f"  ✅ X_train_main: {X_train_main.shape}")
if 'X_val' in locals():
    print(f"  ✅ X_val: {X_val.shape}")
if 'X_train' in locals():
    print(f"  ✅ X_train: {X_train_main.shape}")
if 'X_test' in locals():
    print(f"  ✅ X_test: {y_val.shape}")

# Check the original data
print(f"\nOriginal data:")
print(f"  datasample: {datasample.shape}")

# Check teacher's test data
print(f"\nTeacher's test data:")
if 'test_data' in locals():
    print(f"  ✅ test_data: {test_data.shape}")
else:
    print(f"  ❌ test_data: Not found")

🔍 CHECKING YOUR DATA:
Variables you have created:
  ✅ X_train_main: (504000, 24)
  ✅ X_val: (126000, 24)

Original data:
  datasample: (630000, 25)

Teacher's test data:
  ✅ test_data: (70000, 24)


In [10]:
print(len(X_train_main))
print(len(X_val))

504000
126000


In [11]:
print(X_train_main.columns)
print(X_train_main.head())

Index(['age', 'alcohol_consumption_per_week',
       'physical_activity_minutes_per_week', 'diet_score',
       'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi',
       'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate',
       'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol',
       'triglycerides', 'gender', 'ethnicity', 'education_level',
       'income_level', 'smoking_status', 'employment_status',
       'family_history_diabetes', 'hypertension_history',
       'cardiovascular_history'],
      dtype='object')
        age  alcohol_consumption_per_week  physical_activity_minutes_per_week  \
337030   61                             2                                  53   
180977   38                             1                                  70   
490907   52                             2                                  92   
399622   51                             2                                  52   
622383   37                             5 

# feature engineering

In [12]:
# def advanced_feature_engineering_final(df):
#     df = df.copy()
    
#     def has_col(col_name):
#         return col_name in df.columns

#     # ============ YOUR EXISTING FEATURES ============
#     # Blood pressure
#     if has_col('systolic_bp') and has_col('diastolic_bp'):
#         df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
#         df['map'] = df['diastolic_bp'] + (df['systolic_bp'] - df['diastolic_bp']) / 3
#         df['bp_risk'] = ((df['systolic_bp'] - 120) / 20) + ((df['diastolic_bp'] - 80) / 10)
#         # NEW: Binary hypertension flag
#         df['hypertension'] = ((df['systolic_bp'] >= 140) | (df['diastolic_bp'] >= 90)).astype(int)

#     # Lipids
#     if has_col('triglycerides') and has_col('hdl_cholesterol'):
#         df['atherogenic_index'] = np.log((df['triglycerides'] + 1e-5) / (df['hdl_cholesterol'] + 1e-5))
#         # NEW: Binary lipid flags
#         df['high_triglycerides'] = (df['triglycerides'] >= 150).astype(int)
#         df['low_hdl'] = (df['hdl_cholesterol'] < 40).astype(int)
#         df['trig_hdl_ratio'] = df['triglycerides'] / (df['hdl_cholesterol'] + 1e-5)
    
#     if has_col('cholesterol_total') and has_col('hdl_cholesterol'):
#         df['ldl_hdl_ratio'] = (df['cholesterol_total'] - df['hdl_cholesterol']) / (df['hdl_cholesterol'] + 1e-5)
#         df['high_total_chol'] = (df['cholesterol_total'] >= 200).astype(int)

#     # Metabolic score
#     if has_col('bmi') and has_col('waist_to_hip_ratio') and has_col('triglycerides') and has_col('hdl_cholesterol'):
#         df['metabolic_score'] = (df['bmi']/25 + df['waist_to_hip_ratio']/0.85 + df['triglycerides']/150 - df['hdl_cholesterol']/50)
#         # NEW: Metabolic syndrome binary
#         df['metabolic_syndrome'] = (
#             (df['bmi'] >= 30).astype(int) +
#             (df['waist_to_hip_ratio'] >= 0.85).astype(int) +
#             (df['triglycerides'] >= 150).astype(int) +
#             (df['hdl_cholesterol'] < 40).astype(int)
#         ).clip(upper=1)

#     # Categories
#     if has_col('bmi'):
#         df['bmi_category'] = pd.cut(df['bmi'], bins=[0, 18.5, 25, 30, 35, 100], labels=[0, 1, 2, 3, 4]).astype(float)
#         df['obese'] = (df['bmi'] >= 30).astype(int)
    
#     if has_col('age'):
#         df['age_category'] = pd.cut(df['age'], bins=[0, 30, 40, 50, 60, 100], labels=[0, 1, 2, 3, 4]).astype(float)
#         df['age_over_50'] = (df['age'] > 50).astype(int)

#     # Optional ratio
#     if has_col('physical_activity_minutes_per_week') and has_col('screen_time_hours_per_day'):
#         df['active_sedentary_ratio'] = (df['physical_activity_minutes_per_week'] / 7) / (df['screen_time_hours_per_day'] + 1)
#         df['sedentary'] = (df['screen_time_hours_per_day'] > 5).astype(int)

#     # ============ NEW: SIMPLE INTERACTIONS ============
#     # These are NEW and CRITICAL
    
#     # 1. Simple interaction
#     if has_col('age') and has_col('bmi'):
#         df['age_times_bmi'] = df['age'] * df['bmi'] / 100
    
#     # 2. BP + Triglycerides
#     if has_col('systolic_bp') and has_col('triglycerides'):
#         df['bp_times_trig'] = df['systolic_bp'] * df['triglycerides'] / 1000
    
#     # 3. Age + BP
#     if has_col('age') and has_col('systolic_bp'):
#         df['age_times_bp'] = df['age'] * df['systolic_bp'] / 1000
    
#     # 4. Family history risk amplifier
#     if has_col('family_history_diabetes') and has_col('age'):
#         df['family_risk'] = df['family_history_diabetes'] * (df['age'] > 40).astype(int)
    
#     return df

# # Application
# print("🔧 Application du Feature Engineering Avancé...")
# # Application
# print("🔧 Application du Feature Engineering Avancé...")

# # Use X_train_main and X_val (NOT X_train and X_test)
# X_train_fe_advanced = advanced_feature_engineering_final(X_train_main)
# X_val_fe_advanced = advanced_feature_engineering_final(X_val)

# print(f"✅ Avant: {X_train_main.shape}")
# print(f"✅ Après: {X_train_fe_advanced.shape}")
# print(f"   Nouvelles features créées: {X_train_fe_advanced.shape[1] - X_train_main.shape[1]}")

In [13]:


print(f"\n📊 Shape avant: {X_train_main.shape}")
print(f"📊 Shape après: {X_train_main.shape}")

# Check critical features
print(f"\n🔑 CHECKING NEW FEATURES:")
check_features = [
    'hypertension', 'high_triglycerides', 'low_hdl', 'obese',
    'age_over_50', 'age_times_bmi', 'bp_times_trig', 'metabolic_syndrome'
]

for feat in check_features:
    if feat in X_train_main.columns:
        print(f"  ✅ {feat}: CREATED")
        # Show sample values
        print(f"     Sample: {X_train_main[feat].head(3).values}")
    else:
        print(f"  ❌ {feat}: MISSING")

print("="*60)


📊 Shape avant: (504000, 24)
📊 Shape après: (504000, 24)

🔑 CHECKING NEW FEATURES:
  ❌ hypertension: MISSING
  ❌ high_triglycerides: MISSING
  ❌ low_hdl: MISSING
  ❌ obese: MISSING
  ❌ age_over_50: MISSING
  ❌ age_times_bmi: MISSING
  ❌ bp_times_trig: MISSING
  ❌ metabolic_syndrome: MISSING


# Preprocessing / scalling / encoding

In [14]:
print("⚙️  Preprocessing avec nouvelles features...")

# Colonnes catégorielles (les mêmes qu'avant)
categorical_cols = [
    'gender', 'ethnicity', 'education_level', 'income_level', 
    'smoking_status', 'employment_status', 'family_history_diabetes',
    'hypertension_history', 'cardiovascular_history'
]

categorical_cols_actual = [col for col in categorical_cols if col in X_train_main.columns]
numeric_cols_actual = [col for col in X_train_main.columns if col not in categorical_cols_actual]

print(f"   Colonnes numériques: {len(numeric_cols_actual)}")
print(f"   Colonnes catégorielles: {len(categorical_cols_actual)}")

# Pipeline
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor_advanced = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols_actual),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols_actual)
], remainder='drop')

# 1️⃣ Fit and transform training data
X_train_processed_advanced = preprocessor_advanced.fit_transform(X_train_main)
feature_names = preprocessor_advanced.get_feature_names_out()
X_train_processed_advanced = pd.DataFrame(X_train_processed_advanced, columns=feature_names)  # type: ignore

# 2️⃣ Transform VALIDATION data (was misnamed "test")
X_val_processed_advanced = preprocessor_advanced.transform(X_val)
X_val_processed_advanced = pd.DataFrame(X_val_processed_advanced, columns=feature_names)  # type: ignore


# ✅ Now both train and test have the same columns and order

print(f"✅ Train processed: {X_train_processed_advanced.shape}")
print(f"✅ Validation processed: {X_val_processed_advanced.shape}\n")


⚙️  Preprocessing avec nouvelles features...
   Colonnes numériques: 15
   Colonnes catégorielles: 9
✅ Train processed: (504000, 45)
✅ Validation processed: (126000, 45)



# Data check

In [15]:
# Add this BEFORE any modeling
print("🔍 Checking data quality...")
print(f"Train shape: {X_train_processed_advanced.shape}")
print(f"Validation shape: {X_val_processed_advanced.shape}")  # NOT X_test

# Check for NaN
print(f"\nNaN in train: {X_train_processed_advanced.isna().sum().sum()}")
print(f"NaN in validation: {X_val_processed_advanced.isna().sum().sum()}")
# Check class balance - use correct variable names
print(f"\nClass balance in y_train_main: {y_train_main.value_counts(normalize=True).round(3)}")
print(f"Class balance in y_val: {y_val.value_counts(normalize=True).round(3)}")

🔍 Checking data quality...
Train shape: (504000, 45)
Validation shape: (126000, 45)

NaN in train: 0
NaN in validation: 0

Class balance in y_train_main: diagnosed_diabetes
1    0.623
0    0.377
Name: proportion, dtype: float64
Class balance in y_val: diagnosed_diabetes
1    0.623
0    0.377
Name: proportion, dtype: float64


# no smote cause it's hurts lightboost

In [16]:
# print("⚖️  Application de SMOTE...")

# smote_advanced = SMOTE(random_state=42)
# X_train_balanced_advanced, y_train_balanced_advanced = smote_advanced.fit_resample(
#     X_train_processed_advanced, y_train
# )

# print(f"✅ Données balancées: {X_train_balanced_advanced.shape}")
# print(f"   Distribution: {pd.Series(y_train_balanced_advanced).value_counts(normalize=True).to_dict()}\n")


# lightGBM

In [17]:
# # ================================================================
# # FINAL ACCURACY-OPTIMIZED LIGHTGBM (NO SMOTE)
# # ================================================================
# import numpy as np
# import lightgbm as lgb
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# print("🚀 FINAL LIGHTGBM — NO SMOTE — ACCURACY OPTIMIZED")
# print("="*70)

# # ================================================================
# # DATA (REAL DATA — NO RESAMPLING)
# # ================================================================
# X_train_final = X_train_processed_advanced
# y_train_final = y_train_main

# X_val_final = X_val_processed_advanced
# y_val_final = y_val

# # ================================================================
# # LIGHTGBM CONFIG (ACCURACY-FOCUSED)
# # ================================================================
# lgb_acc = lgb.LGBMClassifier(
#     n_estimators=350,
#     max_depth=8,
#     learning_rate=0.03,
#     num_leaves=32,
#     min_child_samples=30,
#     subsample=0.9,
#     colsample_bytree=0.9,
#     reg_alpha=0.5,
#     reg_lambda=2.0,
#     class_weight=None,      # 🚫 NO class_weight, NO SMOTE
#     random_state=42,
#     n_jobs=-1,
#     verbose=-1
# )

# # ================================================================
# # TRAIN
# # ================================================================
# print("⏳ Training LightGBM (real data)...")
# lgb_acc.fit(X_train_final, y_train_final)
# print("✅ Training completed")

# # ================================================================
# # PROBABILITIES
# # ================================================================
# y_proba_train = lgb_acc.predict_proba(X_train_final)[:, 1]  # train
# y_proba_val   = lgb_acc.predict_proba(X_val_final)[:, 1]    # validation

# # ================================================================
# # MICRO THRESHOLD SEARCH (VALIDATION ONLY)
# # ================================================================
# best_acc = 0
# best_threshold = 0.5

# for t in np.arange(0.46, 0.52, 0.001):
#     preds = (y_proba_val >= t).astype(int)
#     acc = accuracy_score(y_val_final, preds)
#     if acc > best_acc:
#         best_acc = acc
#         best_threshold = t

# print(f"🎯 Best threshold: {best_threshold:.3f}")
# print(f"🎯 Best validation accuracy: {best_acc*100:.2f}%")

# # ================================================================
# # FINAL PREDICTIONS
# # ================================================================
# y_pred_train = (y_proba_train >= best_threshold).astype(int)
# y_pred_val   = (y_proba_val   >= best_threshold).astype(int)

# # ================================================================
# # EVALUATION FUNCTION
# # ================================================================
# def evaluate(y_true, y_pred, name):
#     print(f"\n📊 {name}")
#     print(f"   Accuracy:  {accuracy_score(y_true, y_pred) * 100:.2f}%")
#     print(f"   Precision: {precision_score(y_true, y_pred) * 100:.2f}%")
#     print(f"   Recall:    {recall_score(y_true, y_pred) * 100:.2f}%")
#     print(f"   F1-score:  {f1_score(y_true, y_pred) * 100:.2f}%")

# # ================================================================
# # EVALUATE
# # ================================================================
# evaluate(y_train_final, y_pred_train, "TRAIN")
# evaluate(y_val_final,   y_pred_val,   "VALIDATION")


In [18]:
# ================================================================
# OPTION 2 — ACCURACY + RECALL FOCUS — LIGHTGBM
# ================================================================
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("🐱‍🏍 FINAL LIGHTGBM — RECALL-FOCUSED — NO SMOTE")
print("="*70)

# ================================================================
# DATA (RAW FEATURES — NO FEATURE ENGINEERING)
# ================================================================
X_train_final = X_train_processed_advanced
y_train_final = y_train_main

X_val_final = X_val_processed_advanced
y_val_final = y_val

# ================================================================
# LIGHTGBM CONFIG — STANDARD SETTINGS
# ================================================================
lgb_model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# ================================================================
# TRAIN
# ================================================================
print("⏳ Training LightGBM (raw data)...")
lgb_model.fit(X_train_final, y_train_final)
print("✅ Training completed")

# ================================================================
# PROBABILITIES
# ================================================================
y_proba_train = lgb_model.predict_proba(X_train_final)[:, 1]
y_proba_val   = lgb_model.predict_proba(X_val_final)[:, 1]

# ================================================================
# THRESHOLD TUNING (OPTIONAL — FOR BETTER RECALL)
# ================================================================
best_acc = 0
best_threshold = 0.5

for t in np.arange(0.45, 0.52, 0.001):
    preds_val = (y_proba_val >= t).astype(int)
    acc = accuracy_score(y_val_final, preds_val)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"🎯 Selected threshold: {best_threshold:.3f}")
print(f"🎯 Validation Accuracy at this threshold: {best_acc*100:.2f}%")

# ================================================================
# FINAL PREDICTIONS
# ================================================================
y_pred_train = (y_proba_train >= best_threshold).astype(int)
y_pred_val   = (y_proba_val   >= best_threshold).astype(int)

# ================================================================
# EVALUATION FUNCTION (INCLUDE RECALL FOCUS)
# ================================================================
def evaluate(y_true, y_pred, name):
    print(f"\n📊 {name}")
    print(f"   Accuracy : {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"   Precision: {precision_score(y_true, y_pred)*100:.2f}%")
    print(f"   Recall   : {recall_score(y_true, y_pred)*100:.2f}% 🔑 important")
    print(f"   F1-score : {f1_score(y_true, y_pred)*100:.2f}%")

# ================================================================
# EVALUATE
# ================================================================
evaluate(y_train_final, y_pred_train, "TRAIN")
evaluate(y_val_final,   y_pred_val,   "VALIDATION")


🐱‍🏍 FINAL LIGHTGBM — RECALL-FOCUSED — NO SMOTE
⏳ Training LightGBM (raw data)...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 314141, number of negative: 189859
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.032715 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1671
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.623296 -> initscore=0.503560
[LightGBM] [Info] Start training from score 0.503560
✅ Training completed
🎯 Selected threshold: 0.513
🎯 Validation Accuracy at this threshold: 68.31%

📊 TRAIN
   Accuracy : 69.14%
   Precision: 71.83%
   Recall   : 83.07% 🔑 important
   F1-score : 77.04%

📊 VALIDATION
   Accuracy : 68.31%
   Precision: 71.22%
   Recall   : 82.48% 🔑

# submission

In [19]:
print("\n🎯 Preparing submission for TEST_DATA...")

# ================================================================
# Make sure 'test_data' has all raw features used in training
# ================================================================
X_test_raw = test_data[ X_train_main.columns ]  # raw columns only

# ================================================================
# Apply the same preprocessing as training
# ================================================================
X_test_processed = preprocessor_advanced.transform(X_test_raw)
feature_names = preprocessor_advanced.get_feature_names_out()
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names)  # type: ignore

# ================================================================
# Predict probabilities with trained LightGBM
# ================================================================
y_proba_test_final = lgb_model.predict_proba(X_test_processed)[:, 1]

# Apply the best threshold found on validation
y_pred_test_final = (y_proba_test_final >= best_threshold).astype(float)

# ================================================================
# Prepare submission DataFrame
# ================================================================
submission = pd.DataFrame({
    'id': np.arange(1, len(y_pred_test_final) + 1),
    'diagnosed_diabetes': y_pred_test_final
})

# Save CSV
submission.to_csv('predictions_new.csv', index=False)
print(f"✅ Submission saved: {len(y_pred_test_final)} rows")
print(submission.head(15))



🎯 Preparing submission for TEST_DATA...


✅ Submission saved: 70000 rows
    id  diagnosed_diabetes
0    1                 1.0
1    2                 1.0
2    3                 0.0
3    4                 1.0
4    5                 0.0
5    6                 1.0
6    7                 0.0
7    8                 1.0
8    9                 0.0
9   10                 1.0
10  11                 1.0
11  12                 1.0
12  13                 1.0
13  14                 0.0
14  15                 1.0


# 3rd best 68,37%

In [21]:
# ================================================================
# OPTION 2 — ACCURACY + RECALL FOCUS — LIGHTGBM
# ================================================================
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("🐱‍🏍 FINAL LIGHTGBM — RECALL-FOCUSED — NO SMOTE")
print("="*70)

# ================================================================
# DATA (RAW FEATURES — NO FEATURE ENGINEERING)
# ================================================================
X_train_final = X_train_processed_advanced
y_train_final = y_train_main

X_val_final = X_val_processed_advanced
y_val_final = y_val

# ================================================================
# LIGHTGBM CONFIG — STANDARD SETTINGS
# ================================================================
lgb_model = lgb.LGBMClassifier(
    n_estimators=800,        # ← Change from 500
    learning_rate=0.03,      # ← Change from 0.05
    max_depth=7,             # ← Change from -1
    num_leaves=50,           # ← Add this
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)

# ================================================================
# TRAIN
# ================================================================
print("⏳ Training LightGBM (raw data)...")
lgb_model.fit(X_train_final, y_train_final)
print("✅ Training completed")

# ================================================================
# PROBABILITIES
# ================================================================
y_proba_train = lgb_model.predict_proba(X_train_final)[:, 1]
y_proba_val   = lgb_model.predict_proba(X_val_final)[:, 1]

# ================================================================
# THRESHOLD TUNING (OPTIONAL — FOR BETTER RECALL)
# ================================================================
best_acc = 0
best_threshold = 0.5

for t in np.arange(0.45, 0.52, 0.001):
    preds_val = (y_proba_val >= t).astype(int)
    acc = accuracy_score(y_val_final, preds_val)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"🎯 Selected threshold: {best_threshold:.3f}")
print(f"🎯 Validation Accuracy at this threshold: {best_acc*100:.2f}%")

# ================================================================
# FINAL PREDICTIONS
# ================================================================
y_pred_train = (y_proba_train >= best_threshold).astype(int)
y_pred_val   = (y_proba_val   >= best_threshold).astype(int)

# ================================================================
# EVALUATION FUNCTION (INCLUDE RECALL FOCUS)
# ================================================================
def evaluate(y_true, y_pred, name):
    print(f"\n📊 {name}")
    print(f"   Accuracy : {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"   Precision: {precision_score(y_true, y_pred)*100:.2f}%")
    print(f"   Recall   : {recall_score(y_true, y_pred)*100:.2f}% 🔑 important")
    print(f"   F1-score : {f1_score(y_true, y_pred)*100:.2f}%")

# ================================================================
# EVALUATE
# ================================================================
evaluate(y_train_final, y_pred_train, "TRAIN")
evaluate(y_val_final,   y_pred_val,   "VALIDATION")


🐱‍🏍 FINAL LIGHTGBM — RECALL-FOCUSED — NO SMOTE
⏳ Training LightGBM (raw data)...
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 314141, number of negative: 189859
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.052100 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1671
[LightGBM] [Info] Number of data points in the train set: 504000, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.623296 -> initscore=0.503560
[LightGBM] [Info] Start training from score 0.503560
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive ga

# The 2nd best test acc 68,44%

In [22]:
# ================================================================
# 🎯 HYPERPARAMETER OPTIMIZATION — NO FEATURE ENGINEERING
# ================================================================
import numpy as np
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("🎯 HYPERPARAMETER GRID SEARCH — TARGET: 68.5%+")
print("="*70)

# ================================================================
# DATA (YOUR ORIGINAL FEATURES — NO ENGINEERING)
# ================================================================
X_train_final = X_train_processed_advanced
y_train_final = y_train_main
X_val_final = X_val_processed_advanced
y_val_final = y_val

# ================================================================
# GRID SEARCH OVER MULTIPLE CONFIGURATIONS
# ================================================================
configs = [
    # Config 1: Deeper trees
    {
        'n_estimators': 1000,
        'learning_rate': 0.02,
        'max_depth': 9,
        'num_leaves': 80,
        'min_child_samples': 30,
        'reg_alpha': 0.05,
        'reg_lambda': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    },
    # Config 2: More regularization
    {
        'n_estimators': 900,
        'learning_rate': 0.025,
        'max_depth': 8,
        'num_leaves': 60,
        'min_child_samples': 40,
        'reg_alpha': 0.2,
        'reg_lambda': 0.2,
        'subsample': 0.75,
        'colsample_bytree': 0.75,
    },
    # Config 3: Balanced approach
    {
        'n_estimators': 1200,
        'learning_rate': 0.015,
        'max_depth': 7,
        'num_leaves': 55,
        'min_child_samples': 25,
        'reg_alpha': 0.1,
        'reg_lambda': 0.15,
        'subsample': 0.85,
        'colsample_bytree': 0.85,
    },
    # Config 4: Fast learner
    {
        'n_estimators': 700,
        'learning_rate': 0.04,
        'max_depth': 8,
        'num_leaves': 70,
        'min_child_samples': 20,
        'reg_alpha': 0.08,
        'reg_lambda': 0.08,
        'subsample': 0.8,
        'colsample_bytree': 0.9,
    },
    # Config 5: Conservative
    {
        'n_estimators': 1000,
        'learning_rate': 0.02,
        'max_depth': 6,
        'num_leaves': 45,
        'min_child_samples': 50,
        'reg_alpha': 0.15,
        'reg_lambda': 0.15,
        'subsample': 0.7,
        'colsample_bytree': 0.7,
    },
]

best_overall_acc = 0
best_config = None
best_model = None
best_threshold = 0.5

for i, config in enumerate(configs, 1):
    print(f"\n{'='*70}")
    print(f"🔧 Testing Config {i}/{len(configs)}")
    print(f"   n_estimators={config['n_estimators']}, lr={config['learning_rate']}, depth={config['max_depth']}")
    
    # Train model
    model = lgb.LGBMClassifier(
        **config,
        scale_pos_weight=189859/314141,
        random_state=42,
        n_jobs=-1,
        force_row_wise=True,
        verbose=-1
    )
    
    model.fit(
        X_train_final, 
        y_train_final,
        eval_set=[(X_val_final, y_val_final)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    # Get probabilities
    y_proba_val = model.predict_proba(X_val_final)[:, 1]
    
    # Find best threshold for this model
    config_best_acc = 0
    config_best_threshold = 0.5
    
    for t in np.arange(0.40, 0.60, 0.0005):
        preds = (y_proba_val >= t).astype(int)
        acc = accuracy_score(y_val_final, preds)
        if acc > config_best_acc:
            config_best_acc = acc
            config_best_threshold = t
    
    print(f"   ✅ Val Accuracy: {config_best_acc*100:.2f}% (threshold={config_best_threshold:.4f})")
    
    # Track best overall
    if config_best_acc > best_overall_acc:
        best_overall_acc = config_best_acc
        best_config = config
        best_model = model
        best_threshold = config_best_threshold
        print(f"   🌟 NEW BEST!")

# ================================================================
# FINAL EVALUATION WITH BEST MODEL
# ================================================================
print(f"\n{'='*70}")
print(f"🏆 BEST CONFIGURATION FOUND")
print(f"{'='*70}")
print(f"Validation Accuracy: {best_overall_acc*100:.2f}%")
print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"\nBest Hyperparameters:")
for key, value in best_config.items():
    print(f"   {key}: {value}")

# Get final predictions
y_proba_train = best_model.predict_proba(X_train_final)[:, 1]
y_proba_val = best_model.predict_proba(X_val_final)[:, 1]

y_pred_train = (y_proba_train >= best_threshold).astype(int)
y_pred_val = (y_proba_val >= best_threshold).astype(int)

# Evaluate
def evaluate(y_true, y_pred, name):
    print(f"\n📊 {name}")
    print(f"   Accuracy : {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"   Precision: {precision_score(y_true, y_pred)*100:.2f}%")
    print(f"   Recall   : {recall_score(y_true, y_pred)*100:.2f}% 🔑")
    print(f"   F1-score : {f1_score(y_true, y_pred)*100:.2f}%")

evaluate(y_train_final, y_pred_train, "TRAIN")
evaluate(y_val_final, y_pred_val, "VALIDATION")

🎯 HYPERPARAMETER GRID SEARCH — TARGET: 68.5%+

🔧 Testing Config 1/5
   n_estimators=1000, lr=0.02, depth=9
   ✅ Val Accuracy: 68.34% (threshold=0.4020)
   🌟 NEW BEST!

🔧 Testing Config 2/5
   n_estimators=900, lr=0.025, depth=8
   ✅ Val Accuracy: 68.30% (threshold=0.4010)

🔧 Testing Config 3/5
   n_estimators=1200, lr=0.015, depth=7
   ✅ Val Accuracy: 68.23% (threshold=0.4010)

🔧 Testing Config 4/5
   n_estimators=700, lr=0.04, depth=8
   ✅ Val Accuracy: 68.44% (threshold=0.4030)
   🌟 NEW BEST!

🔧 Testing Config 5/5
   n_estimators=1000, lr=0.02, depth=6
   ✅ Val Accuracy: 68.17% (threshold=0.4045)

🏆 BEST CONFIGURATION FOUND
Validation Accuracy: 68.44%
Optimal Threshold: 0.4030

Best Hyperparameters:
   n_estimators: 700
   learning_rate: 0.04
   max_depth: 8
   num_leaves: 70
   min_child_samples: 20
   reg_alpha: 0.08
   reg_lambda: 0.08
   subsample: 0.8
   colsample_bytree: 0.9

📊 TRAIN
   Accuracy : 70.44%
   Precision: 73.48%
   Recall   : 82.27% 🔑
   F1-score : 77.63%

📊 VALIDA

# The best submision

In [46]:
# ================================================================
# 🏆 FINAL SUBMISSION WORKFLOW
# ================================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("🏆 TRAINING AND SELECTING BEST MODEL BASED ON VALIDATION ACCURACY")
print("="*70)

# ================================================================
# DATA
# ================================================================
X_train_final = X_train_processed_advanced
y_train_final = y_train_main
X_val_final = X_val_processed_advanced
y_val_final = y_val

# ================================================================
# DEFINE CANDIDATE MODELS
# ================================================================
candidates = {
    'Config 4 (Original Best)': {
        'n_estimators': 700,
        'learning_rate': 0.04,
        'max_depth': 8,
        'num_leaves': 70,
        'min_child_samples': 20,
        'reg_alpha': 0.08,
        'reg_lambda': 0.08,
        'subsample': 0.8,
        'colsample_bytree': 0.9,
    },
    'Config 4 Variant A (More trees)': {
        'n_estimators': 750,
        'learning_rate': 0.04,
        'max_depth': 8,
        'num_leaves': 70,
        'min_child_samples': 20,
        'reg_alpha': 0.08,
        'reg_lambda': 0.08,
        'subsample': 0.8,
        'colsample_bytree': 0.9,
    },
    'Config 4 Variant B (Slower LR)': {
        'n_estimators': 800,
        'learning_rate': 0.035,
        'max_depth': 8,
        'num_leaves': 70,
        'min_child_samples': 20,
        'reg_alpha': 0.08,
        'reg_lambda': 0.08,
        'subsample': 0.8,
        'colsample_bytree': 0.9,
    },
    'Config 4 Variant C (More leaves)': {
        'n_estimators': 700,
        'learning_rate': 0.04,
        'max_depth': 8,
        'num_leaves': 75,
        'min_child_samples': 20,
        'reg_alpha': 0.08,
        'reg_lambda': 0.08,
        'subsample': 0.8,
        'colsample_bytree': 0.9,
    },
    'Config 1 (Deep trees)': {
        'n_estimators': 1000,
        'learning_rate': 0.02,
        'max_depth': 9,
        'num_leaves': 80,
        'min_child_samples': 30,
        'reg_alpha': 0.05,
        'reg_lambda': 0.05,
        'subsample': 0.8,
        'colsample_bytree': 0.8,
    },
}

results = []

# ================================================================
# TRAIN MODELS AND SELECT BEST THRESHOLD
# ================================================================
for name, config in candidates.items():
    print(f"\n{'='*70}")
    print(f"🔧 Training model: {name}")
    
    model = lgb.LGBMClassifier(
        **config,
        scale_pos_weight=189859/314141,
        random_state=42,
        n_jobs=-1,
        force_row_wise=True,
        verbose=-1
    )
    
    model.fit(
        X_train_final, y_train_final,
        eval_set=[(X_val_final, y_val_final)],
        callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
    )
    
    # Validation probabilities
    y_proba_val = model.predict_proba(X_val_final)[:, 1]
    
    # Fine-grained threshold search
    best_acc = 0
    best_threshold = 0.5
    for t in np.arange(0.38, 0.58, 0.0002):
        preds = (y_proba_val >= t).astype(int)
        acc = accuracy_score(y_val_final, preds)
        if acc > best_acc:
            best_acc = acc
            best_threshold = t
    
    results.append({
        'name': name,
        'model': model,
        'val_acc': best_acc,
        'threshold': best_threshold
    })
    print(f"   ✅ Val Accuracy: {best_acc*100:.2f}% at threshold {best_threshold:.4f}")

# ================================================================
# SELECT BEST MODEL BY VALIDATION ACCURACY
# ================================================================
best_result = max(results, key=lambda x: x['val_acc'])
best_model = best_result['model']
best_threshold = best_result['threshold']

print("\n" + "="*70)
print("🏆 BEST MODEL SELECTED")
print("="*70)
print(f"Model: {best_result['name']}")
print(f"Validation Accuracy: {best_result['val_acc']*100:.2f}%")
print(f"Threshold used: {best_threshold:.4f}")

# ================================================================
# GENERATE TEST PREDICTIONS
# ================================================================
X_test_raw = test_data[X_train_main.columns]  # Use same columns as training
X_test_processed = preprocessor_advanced.transform(X_test_raw)
feature_names = preprocessor_advanced.get_feature_names_out()
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names)

y_test_proba = best_model.predict_proba(X_test_processed)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(float)

# Create submission
submission = pd.DataFrame({
    'id': np.arange(1, len(y_test_pred) + 1),
    'diagnosed_diabetes': y_test_pred
})
submission.to_csv('submission_best.csv', index=False)
print("✅ Submission saved: submission_best.csv")
print(f"Shape: {submission.shape}")
print(submission.head(10))

# ================================================================
# OPTIONAL: Save model for future use
# ================================================================
import pickle
with open('best_model.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'threshold': best_threshold,
        'val_acc': best_result['val_acc']
    }, f)
print("✅ Model saved: best_model.pkl")

# ================================================================
# VALIDATION METRICS
# ================================================================
y_pred_val = (best_model.predict_proba(X_val_final)[:, 1] >= best_threshold).astype(int)

def evaluate(y_true, y_pred, name):
    print(f"\n📊 {name}")
    print(f"   Accuracy : {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"   Precision: {precision_score(y_true, y_pred)*100:.2f}%")
    print(f"   Recall   : {recall_score(y_true, y_pred)*100:.2f}% 🔑")
    print(f"   F1-score : {f1_score(y_true, y_pred)*100:.2f}%")

evaluate(y_val_final, y_pred_val, "VALIDATION")

# ================================================================
# OPTIONAL: Compare with previous submission
# ================================================================
# prev_submission = pd.read_csv('predictions_old.csv')  # Uncomment if you have old submission
# match = (prev_submission['diagnosed_diabetes'].values == submission['diagnosed_diabetes'].values).mean()
# print(f"\n📊 Fraction of identical predictions with previous submission: {match*100:.2f}%")


🏆 TRAINING AND SELECTING BEST MODEL BASED ON VALIDATION ACCURACY

🔧 Training model: Config 4 (Original Best)
   ✅ Val Accuracy: 68.45% at threshold 0.3922

🔧 Training model: Config 4 Variant A (More trees)
   ✅ Val Accuracy: 68.47% at threshold 0.3922

🔧 Training model: Config 4 Variant B (Slower LR)
   ✅ Val Accuracy: 68.41% at threshold 0.3906

🔧 Training model: Config 4 Variant C (More leaves)
   ✅ Val Accuracy: 68.36% at threshold 0.4070

🔧 Training model: Config 1 (Deep trees)
   ✅ Val Accuracy: 68.37% at threshold 0.3968

🏆 BEST MODEL SELECTED
Model: Config 4 Variant A (More trees)
Validation Accuracy: 68.47%
Threshold used: 0.3922
✅ Submission saved: submission_best.csv
Shape: (70000, 2)
   id  diagnosed_diabetes
0   1                 1.0
1   2                 1.0
2   3                 0.0
3   4                 1.0
4   5                 0.0
5   6                 1.0
6   7                 0.0
7   8                 1.0
8   9                 0.0
9  10                 1.0
✅ Model sa

In [38]:
# ================================================================
# 🔍 DATA ANALYSIS — FIND WHAT'S LIMITING PERFORMANCE
# ================================================================
import pandas as pd
import numpy as np

print("🔍 ANALYZING DATA QUALITY")
print("="*70)

# 1. Check feature statistics
print("\n📊 FEATURE STATISTICS")
print(f"Number of features: {X_train_processed_advanced.shape[1]}")
print(f"Training samples: {X_train_processed_advanced.shape[0]}")
print(f"Validation samples: {X_val_processed_advanced.shape[0]}")

# 2. Check class balance
print("\n⚖️ CLASS DISTRIBUTION")
print(f"Train - Class 0: {(y_train_main == 0).sum()} ({(y_train_main == 0).sum()/len(y_train_main)*100:.1f}%)")
print(f"Train - Class 1: {(y_train_main == 1).sum()} ({(y_train_main == 1).sum()/len(y_train_main)*100:.1f}%)")
print(f"Val   - Class 0: {(y_val == 0).sum()} ({(y_val == 0).sum()/len(y_val)*100:.1f}%)")
print(f"Val   - Class 1: {(y_val == 1).sum()} ({(y_val == 1).sum()/len(y_val)*100:.1f}%)")

# 3. Check for data leakage or issues
print("\n🔍 CHECKING FOR ISSUES")

# Check if train/val distributions are similar
from scipy.stats import ks_2samp

numeric_cols = X_train_processed_advanced.select_dtypes(include=[np.number]).columns[:10]
different_distributions = []

for col in numeric_cols:
    stat, p_value = ks_2samp(X_train_processed_advanced[col], X_val_processed_advanced[col])
    if p_value < 0.01:  # Significantly different
        different_distributions.append((col, p_value))

if different_distributions:
    print(f"⚠️ Found {len(different_distributions)} features with different train/val distributions:")
    for col, p in different_distributions[:5]:
        print(f"   - {col}: p={p:.4f}")
else:
    print("✅ Train/val distributions look similar")

# 4. Check feature importance to see if you're missing key features
print("\n🔝 TOP 15 MOST IMPORTANT FEATURES")
feature_importance = pd.DataFrame({
    'feature': X_train_processed_advanced.columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

for idx, row in feature_importance.head(15).iterrows():
    print(f"   {row['feature']}: {row['importance']:.0f}")

# Check if top features dominate
top_10_importance = feature_importance.head(10)['importance'].sum()
total_importance = feature_importance['importance'].sum()
print(f"\n📊 Top 10 features account for {top_10_importance/total_importance*100:.1f}% of importance")

if top_10_importance/total_importance > 0.8:
    print("⚠️ Model heavily relies on few features - might be missing important interactions")

🔍 ANALYZING DATA QUALITY

📊 FEATURE STATISTICS
Number of features: 45
Training samples: 504000
Validation samples: 126000

⚖️ CLASS DISTRIBUTION
Train - Class 0: 189859 (37.7%)
Train - Class 1: 314141 (62.3%)
Val   - Class 0: 47465 (37.7%)
Val   - Class 1: 78535 (62.3%)

🔍 CHECKING FOR ISSUES
✅ Train/val distributions look similar

🔝 TOP 15 MOST IMPORTANT FEATURES
   num__physical_activity_minutes_per_week: 7658
   num__triglycerides: 4307
   num__bmi: 3632
   num__age: 3279
   num__ldl_cholesterol: 2883
   num__cholesterol_total: 2838
   num__diet_score: 2703
   num__screen_time_hours_per_day: 2696
   num__systolic_bp: 2678
   num__heart_rate: 2437
   num__hdl_cholesterol: 2384
   num__sleep_hours_per_day: 2267
   num__diastolic_bp: 2107
   num__waist_to_hip_ratio: 1755
   num__alcohol_consumption_per_week: 571

📊 Top 10 features account for 73.5% of importance


In [41]:
# ================================================================
# 🐱 CATBOOST — DIFFERENT ALGORITHM
# ================================================================
from catboost import CatBoostClassifier
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

print("🐱 TRAINING CATBOOST MODEL")
print("="*70)

# ================================================================
# DATA
# ================================================================
X_train_final = X_train_processed_advanced
y_train_final = y_train_main
X_val_final = X_val_processed_advanced
y_val_final = y_val

# ================================================================
# CATBOOST CONFIGS TO TEST
# ================================================================
catboost_configs = [
    {
        'name': 'CatBoost A (Balanced)',
        'iterations': 1000,
        'learning_rate': 0.05,
        'depth': 8,
        'l2_leaf_reg': 3,
        'random_strength': 1,
        'bagging_temperature': 1,
    },
    {
        'name': 'CatBoost B (Deep)',
        'iterations': 1200,
        'learning_rate': 0.03,
        'depth': 10,
        'l2_leaf_reg': 5,
        'random_strength': 1.5,
        'bagging_temperature': 0.8,
    },
    {
        'name': 'CatBoost C (Fast)',
        'iterations': 800,
        'learning_rate': 0.07,
        'depth': 7,
        'l2_leaf_reg': 2,
        'random_strength': 0.5,
        'bagging_temperature': 1.2,
    },
]

best_catboost_acc = 0
best_catboost_model = None
best_catboost_threshold = 0.5
best_catboost_name = ""

for config in catboost_configs:
    print(f"\n🔧 Training: {config['name']}")
    
    name = config.pop('name')
    
    model = CatBoostClassifier(
        **config,
        scale_pos_weight=189859/314141,
        random_seed=42,
        verbose=False,
        thread_count=-1
    )
    
    model.fit(X_train_final, y_train_final)
    
    y_proba_val = model.predict_proba(X_val_final)[:, 1]
    
    # Find best threshold
    best_acc = 0
    best_threshold = 0.5
    for t in np.arange(0.35, 0.55, 0.0002):
        preds = (y_proba_val >= t).astype(int)
        acc = accuracy_score(y_val_final, preds)
        if acc > best_acc:
            best_acc = acc
            best_threshold = t
    
    print(f"   Val Accuracy: {best_acc*100:.2f}% @ threshold {best_threshold:.4f}")
    
    if best_acc > best_catboost_acc:
        best_catboost_acc = best_acc
        best_catboost_model = model
        best_catboost_threshold = best_threshold
        best_catboost_name = name
        print(f"   🌟 NEW BEST!")

print(f"\n{'='*70}")
print(f"🏆 BEST CATBOOST MODEL: {best_catboost_name}")
print(f"   Validation Accuracy: {best_catboost_acc*100:.2f}%")
print(f"   Threshold: {best_catboost_threshold:.4f}")

🐱 TRAINING CATBOOST MODEL

🔧 Training: CatBoost A (Balanced)
   Val Accuracy: 68.26% @ threshold 0.3960
   🌟 NEW BEST!

🔧 Training: CatBoost B (Deep)
   Val Accuracy: 68.25% @ threshold 0.3848

🔧 Training: CatBoost C (Fast)
   Val Accuracy: 68.31% @ threshold 0.4056
   🌟 NEW BEST!

🏆 BEST CATBOOST MODEL: CatBoost C (Fast)
   Validation Accuracy: 68.31%
   Threshold: 0.4056


In [44]:
# ================================================================
# 🚀 CATBOOST + STACKING — OBJECTIF: 68.50%+
# ================================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

print("🚀 TESTING CATBOOST + STACKING — TARGET: 68.50%+")
print("="*70)

# ================================================================
# DATA
# ================================================================
X_train_final = X_train_processed_advanced
y_train_final = y_train_main
X_val_final = X_val_processed_advanced
y_val_final = y_val

all_results = []

# ================================================================
# 1. TON MEILLEUR LIGHTGBM (BASELINE: 68.47%)
# ================================================================
print("\n1️⃣ LightGBM (ton meilleur) — Baseline: 68.47%")
lgb_best = lgb.LGBMClassifier(
    n_estimators=750,
    learning_rate=0.04,
    max_depth=8,
    num_leaves=70,
    min_child_samples=20,
    reg_alpha=0.08,
    reg_lambda=0.08,
    subsample=0.8,
    colsample_bytree=0.9,
    scale_pos_weight=189859/314141,
    random_state=42,
    n_jobs=-1,
    force_row_wise=True,
    verbose=-1
)
lgb_best.fit(X_train_final, y_train_final)

all_results.append({
    'name': 'LightGBM Best',
    'model': lgb_best,
    'val_acc': 0.6847,
    'threshold': 0.3922
})

# ================================================================
# 2. CATBOOST (SOUVENT MEILLEUR QUE LIGHTGBM)
# ================================================================
print("\n2️⃣ CatBoost — Testing 5 configs...")

catboost_configs = [
    {'name': 'CatBoost A', 'iterations': 800, 'learning_rate': 0.05, 'depth': 8, 'l2_leaf_reg': 3},
    {'name': 'CatBoost B', 'iterations': 1000, 'learning_rate': 0.04, 'depth': 9, 'l2_leaf_reg': 4},
    {'name': 'CatBoost C', 'iterations': 700, 'learning_rate': 0.06, 'depth': 7, 'l2_leaf_reg': 2},
    {'name': 'CatBoost D', 'iterations': 900, 'learning_rate': 0.045, 'depth': 8, 'l2_leaf_reg': 3.5},
    {'name': 'CatBoost E', 'iterations': 1200, 'learning_rate': 0.03, 'depth': 9, 'l2_leaf_reg': 5},
]

for config in catboost_configs:
    name = config.pop('name')
    print(f"   Training {name}...")
    
    model = CatBoostClassifier(
        **config,
        auto_class_weights='Balanced',
        random_seed=42,
        verbose=False,
        thread_count=-1
    )
    
    model.fit(X_train_final, y_train_final)
    y_proba_val = model.predict_proba(X_val_final)[:, 1]
    
    # Find best threshold
    best_acc = 0
    best_threshold = 0.5
    for t in np.arange(0.35, 0.55, 0.0001):
        preds = (y_proba_val >= t).astype(int)
        acc = accuracy_score(y_val_final, preds)
        if acc > best_acc:
            best_acc = acc
            best_threshold = t
    
    all_results.append({
        'name': name,
        'model': model,
        'val_acc': best_acc,
        'threshold': best_threshold
    })
    
    print(f"      Val Acc: {best_acc*100:.2f}% @ {best_threshold:.4f}")

# ================================================================
# 3. STACKING ENSEMBLE (META-MODEL)
# ================================================================
print("\n3️⃣ Stacking Ensemble...")

# Base models with diversity
base_models = [
    ('lgb1', lgb.LGBMClassifier(
        n_estimators=750, learning_rate=0.04, max_depth=8, num_leaves=70,
        min_child_samples=20, reg_alpha=0.08, reg_lambda=0.08,
        subsample=0.8, colsample_bytree=0.9,
        scale_pos_weight=189859/314141, random_state=42,
        n_jobs=-1, force_row_wise=True, verbose=-1
    )),
    ('lgb2', lgb.LGBMClassifier(
        n_estimators=1000, learning_rate=0.02, max_depth=9, num_leaves=80,
        min_child_samples=30, reg_alpha=0.05, reg_lambda=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=189859/314141, random_state=123,
        n_jobs=-1, force_row_wise=True, verbose=-1
    )),
    ('lgb3', lgb.LGBMClassifier(
        n_estimators=600, learning_rate=0.05, max_depth=7, num_leaves=60,
        min_child_samples=25, reg_alpha=0.10, reg_lambda=0.10,
        subsample=0.75, colsample_bytree=0.85,
        scale_pos_weight=189859/314141, random_state=456,
        n_jobs=-1, force_row_wise=True, verbose=-1
    )),
]

stacking_model = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    cv=3,
    n_jobs=-1
)

print("   Training stacking (this takes ~5 min)...")
stacking_model.fit(X_train_final, y_train_final)

y_proba_val_stack = stacking_model.predict_proba(X_val_final)[:, 1]

# Find best threshold
best_stack_acc = 0
best_stack_threshold = 0.5
for t in np.arange(0.35, 0.55, 0.0001):
    preds = (y_proba_val_stack >= t).astype(int)
    acc = accuracy_score(y_val_final, preds)
    if acc > best_stack_acc:
        best_stack_acc = acc
        best_stack_threshold = t

all_results.append({
    'name': 'Stacking Ensemble',
    'model': stacking_model,
    'val_acc': best_stack_acc,
    'threshold': best_stack_threshold
})

print(f"   Val Acc: {best_stack_acc*100:.2f}% @ {best_stack_threshold:.4f}")

# ================================================================
# 4. WEIGHTED AVERAGE ENSEMBLE
# ================================================================
print("\n4️⃣ Weighted Average Ensemble...")

# Get top 3 models
top_3 = sorted(all_results, key=lambda x: x['val_acc'], reverse=True)[:3]

print(f"   Top 3 models:")
for i, r in enumerate(top_3, 1):
    print(f"      {i}. {r['name']} ({r['val_acc']*100:.2f}%)")

# Average predictions
avg_proba_val = np.mean([
    r['model'].predict_proba(X_val_final)[:, 1] 
    for r in top_3
], axis=0)

# Find best threshold
best_avg_acc = 0
best_avg_threshold = 0.5
for t in np.arange(0.35, 0.55, 0.0001):
    preds = (avg_proba_val >= t).astype(int)
    acc = accuracy_score(y_val_final, preds)
    if acc > best_avg_acc:
        best_avg_acc = acc
        best_avg_threshold = t

all_results.append({
    'name': 'Weighted Avg (Top 3)',
    'models': top_3,
    'val_acc': best_avg_acc,
    'threshold': best_avg_threshold,
    'is_ensemble': True
})

print(f"   Val Acc: {best_avg_acc*100:.2f}% @ {best_avg_threshold:.4f}")

# ================================================================
# RANKING FINAL
# ================================================================
print("\n" + "="*70)
print("🏆 RANKING FINAL — TOUS LES MODÈLES")
print("="*70)

all_results_sorted = sorted(all_results, key=lambda x: x['val_acc'], reverse=True)

for i, r in enumerate(all_results_sorted, 1):
    status = "🎉 OBJECTIF ATTEINT!" if r['val_acc'] >= 0.6850 else ""
    print(f"{i}. {r['name']:35s} | {r['val_acc']*100:.2f}% @ {r['threshold']:.4f} {status}")

# ================================================================
# SÉLECTION DU MEILLEUR
# ================================================================
absolute_best = all_results_sorted[0]

print(f"\n{'='*70}")
print(f"🏆 MEILLEUR MODÈLE: {absolute_best['name']}")
print(f"   Validation Accuracy: {absolute_best['val_acc']*100:.2f}%")
print(f"   Threshold: {absolute_best['threshold']:.4f}")

if absolute_best['val_acc'] >= 0.6850:
    print(f"\n✅ OBJECTIF 68.50%+ ATTEINT!")
    improvement = (absolute_best['val_acc'] - 0.6847) * 100
    print(f"   Amélioration: +{improvement:.2f}% vs LightGBM")
else:
    gap = (0.6850 - absolute_best['val_acc']) * 100
    print(f"\n⚠️ Encore {gap:.2f}% pour atteindre 68.50%")

# ================================================================
# GÉNÉRER PRÉDICTIONS TEST
# ================================================================
print(f"\n{'='*70}")
print("📤 GÉNÉRATION DES PRÉDICTIONS TEST")
print(f"{'='*70}")

X_test_raw = test_data[X_train_main.columns]
X_test_processed = preprocessor_advanced.transform(X_test_raw)
feature_names = preprocessor_advanced.get_feature_names_out()
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names)

# Generate predictions based on best model type
if 'is_ensemble' in absolute_best and absolute_best['is_ensemble']:
    # Weighted average ensemble
    y_test_proba = np.mean([
        r['model'].predict_proba(X_test_processed)[:, 1] 
        for r in absolute_best['models']
    ], axis=0)
else:
    # Single model
    y_test_proba = absolute_best['model'].predict_proba(X_test_processed)[:, 1]

y_test_pred = (y_test_proba >= absolute_best['threshold']).astype(float)

# Create submission
submission = pd.DataFrame({
    'id': np.arange(1, len(y_test_pred) + 1),
    'diagnosed_diabetes': y_test_pred
})

submission.to_csv('submission_FINAL_6850.csv', index=False)

print(f"✅ Submission saved: submission_FINAL_6850.csv")
print(f"   Shape: {submission.shape}")
print(f"   Class distribution:")
print(f"   - Class 0: {(y_test_pred == 0).sum()} ({(y_test_pred == 0).sum()/len(y_test_pred)*100:.1f}%)")
print(f"   - Class 1: {(y_test_pred == 1).sum()} ({(y_test_pred == 1).sum()/len(y_test_pred)*100:.1f}%)")

# ================================================================
# VALIDATION METRICS
# ================================================================
if 'is_ensemble' in absolute_best and absolute_best['is_ensemble']:
    y_pred_val = (avg_proba_val >= absolute_best['threshold']).astype(int)
else:
    y_pred_val = (absolute_best['model'].predict_proba(X_val_final)[:, 1] >= absolute_best['threshold']).astype(int)

print(f"\n📊 VALIDATION METRICS")
print(f"   Accuracy : {accuracy_score(y_val_final, y_pred_val)*100:.2f}%")
print(f"   Precision: {precision_score(y_val_final, y_pred_val)*100:.2f}%")
print(f"   Recall   : {recall_score(y_val_final, y_pred_val)*100:.2f}% 🔑")
print(f"   F1-score : {f1_score(y_val_final, y_pred_val)*100:.2f}%")

print(f"\n{'='*70}")
print("🎯 RÉSUMÉ")
print(f"{'='*70}")
print(f"Baseline (LightGBM):      68.47%")
print(f"Meilleur modèle:          {absolute_best['val_acc']*100:.2f}%")
print(f"Fichier de submission:    submission_FINAL_6850.csv")
print(f"{'='*70}")

🚀 TESTING CATBOOST + STACKING — TARGET: 68.50%+

1️⃣ LightGBM (ton meilleur) — Baseline: 68.47%

2️⃣ CatBoost — Testing 5 configs...
   Training CatBoost A...
      Val Acc: 68.24% @ 0.3808
   Training CatBoost B...
      Val Acc: 68.22% @ 0.3726
   Training CatBoost C...
      Val Acc: 68.25% @ 0.3946
   Training CatBoost D...
      Val Acc: 68.23% @ 0.3725
   Training CatBoost E...
      Val Acc: 68.20% @ 0.3829

3️⃣ Stacking Ensemble...
   Training stacking (this takes ~5 min)...
   Val Acc: 68.44% @ 0.3722

4️⃣ Weighted Average Ensemble...
   Top 3 models:
      1. LightGBM Best (68.47%)
      2. Stacking Ensemble (68.44%)
      3. CatBoost C (68.25%)
   Val Acc: 68.44% @ 0.3926

🏆 RANKING FINAL — TOUS LES MODÈLES
1. LightGBM Best                       | 68.47% @ 0.3922 
2. Stacking Ensemble                   | 68.44% @ 0.3722 
3. Weighted Avg (Top 3)                | 68.44% @ 0.3926 
4. CatBoost C                          | 68.25% @ 0.3946 
5. CatBoost A                          |

In [47]:
# ================================================================
# 🏆 AUGMENTED LIGHTGBM FINAL SUBMISSION WORKFLOW
# ================================================================
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pickle

print("🏆 TRAINING AND SELECTING BEST MODEL BASED ON VALIDATION ACCURACY")
print("="*70)

# ================================================================
# DATA
# ================================================================
X_train_final = X_train_processed_advanced
y_train_final = y_train_main
X_val_final = X_val_processed_advanced
y_val_final = y_val

# ================================================================
# AUGMENTED HYPERPARAMETER CANDIDATES
# ================================================================
candidates = {
    'Variant A (Original + tweak)': {
        'n_estimators': 760, 'learning_rate': 0.041, 'max_depth': 8, 'num_leaves': 71,
        'min_child_samples': 20, 'reg_alpha': 0.082, 'reg_lambda': 0.082,
        'subsample': 0.81, 'colsample_bytree': 0.905
    },
    'Variant B (More Trees + slightly lower LR)': {
        'n_estimators': 820, 'learning_rate': 0.038, 'max_depth': 8, 'num_leaves': 72,
        'min_child_samples': 20, 'reg_alpha': 0.085, 'reg_lambda': 0.085,
        'subsample': 0.82, 'colsample_bytree': 0.91
    },
    'Variant C (Deeper + more leaves)': {
        'n_estimators': 800, 'learning_rate': 0.037, 'max_depth': 9, 'num_leaves': 78,
        'min_child_samples': 22, 'reg_alpha': 0.09, 'reg_lambda': 0.09,
        'subsample': 0.83, 'colsample_bytree': 0.88
    },
    'Variant D (Aggressive Sampling)': {
        'n_estimators': 780, 'learning_rate': 0.042, 'max_depth': 8, 'num_leaves': 73,
        'min_child_samples': 20, 'reg_alpha': 0.08, 'reg_lambda': 0.08,
        'subsample': 0.87, 'colsample_bytree': 0.92
    },
    'Variant E (More regularization)': {
        'n_estimators': 800, 'learning_rate': 0.035, 'max_depth': 8, 'num_leaves': 70,
        'min_child_samples': 25, 'reg_alpha': 0.1, 'reg_lambda': 0.1,
        'subsample': 0.8, 'colsample_bytree': 0.9
    },
}

# ================================================================
# TRAIN MODELS AND FIND BEST THRESHOLD
# ================================================================
results = []

for name, config in candidates.items():
    print(f"\n{'='*70}")
    print(f"🔧 Training model: {name}")

    model = lgb.LGBMClassifier(
        **config,
        scale_pos_weight=189859/314141,
        random_state=42,
        n_jobs=-1,
        force_row_wise=True,
        verbose=-1
    )

    model.fit(
        X_train_final, y_train_final,
        eval_set=[(X_val_final, y_val_final)],
        callbacks=[lgb.early_stopping(stopping_rounds=150, verbose=False)]
    )

    # Validation probabilities
    y_proba_val = model.predict_proba(X_val_final)[:, 1]

    # Fine-grained threshold search
    best_acc = 0
    best_threshold = 0.5
    for t in np.arange(0.38, 0.58, 0.0001):
        preds = (y_proba_val >= t).astype(int)
        acc = accuracy_score(y_val_final, preds)
        if acc > best_acc:
            best_acc = acc
            best_threshold = t

    results.append({'name': name, 'model': model, 'val_acc': best_acc, 'threshold': best_threshold})
    print(f"   ✅ Val Accuracy: {best_acc*100:.2f}% at threshold {best_threshold:.4f}")

# ================================================================
# SELECT BEST MODEL
# ================================================================
best_result = max(results, key=lambda x: x['val_acc'])
best_model = best_result['model']
best_threshold = best_result['threshold']

print("\n" + "="*70)
print(f"🏆 BEST MODEL: {best_result['name']} - Val Accuracy: {best_result['val_acc']*100:.2f}%")
print(f"Threshold: {best_threshold:.4f}")

# ================================================================
# VALIDATION METRICS
# ================================================================
y_pred_val = (best_model.predict_proba(X_val_final)[:, 1] >= best_threshold).astype(int)

def evaluate(y_true, y_pred, name):
    print(f"\n📊 {name}")
    print(f"   Accuracy : {accuracy_score(y_true, y_pred)*100:.2f}%")
    print(f"   Precision: {precision_score(y_true, y_pred)*100:.2f}%")
    print(f"   Recall   : {recall_score(y_true, y_pred)*100:.2f}% 🔑")
    print(f"   F1-score : {f1_score(y_true, y_pred)*100:.2f}%")

evaluate(y_val_final, y_pred_val, "VALIDATION")

# ================================================================
# TEST PREDICTIONS
# ================================================================
X_test_raw = test_data[X_train_main.columns]
X_test_processed = preprocessor_advanced.transform(X_test_raw)
feature_names = preprocessor_advanced.get_feature_names_out()
X_test_processed = pd.DataFrame(X_test_processed, columns=feature_names)

y_test_proba = best_model.predict_proba(X_test_processed)[:, 1]
y_test_pred = (y_test_proba >= best_threshold).astype(float)

# ================================================================
# CREATE SUBMISSION FILE
# ================================================================
submission = pd.DataFrame({
    'id': np.arange(1, len(y_test_pred)+1),
    'diagnosed_diabetes': y_test_pred
})
submission.to_csv('submission_best_augmentednew.csv', index=False)
print("\n✅ Submission saved: submission_best_augmentednew.csv")
print(f"Shape: {submission.shape}")
print(submission.head(10))

# ================================================================
# SAVE MODEL
# ================================================================
with open('best_model_augmented.pkl', 'wb') as f:
    pickle.dump({
        'model': best_model,
        'threshold': best_threshold,
        'val_acc': best_result['val_acc'],
        'config': candidates[best_result['name']]
    }, f)
print("✅ Model saved: best_model_augmented.pkl")


🏆 TRAINING AND SELECTING BEST MODEL BASED ON VALIDATION ACCURACY

🔧 Training model: Variant A (Original + tweak)
   ✅ Val Accuracy: 68.41% at threshold 0.3833

🔧 Training model: Variant B (More Trees + slightly lower LR)
   ✅ Val Accuracy: 68.46% at threshold 0.3919

🔧 Training model: Variant C (Deeper + more leaves)
   ✅ Val Accuracy: 68.39% at threshold 0.3825

🔧 Training model: Variant D (Aggressive Sampling)
   ✅ Val Accuracy: 68.42% at threshold 0.4010

🔧 Training model: Variant E (More regularization)
   ✅ Val Accuracy: 68.37% at threshold 0.3958

🏆 BEST MODEL: Variant B (More Trees + slightly lower LR) - Val Accuracy: 68.46%
Threshold: 0.3919

📊 VALIDATION
   Accuracy : 68.46%
   Precision: 71.35%
   Recall   : 82.54% 🔑
   F1-score : 76.54%

✅ Submission saved: submission_best_augmentednew.csv
Shape: (70000, 2)
   id  diagnosed_diabetes
0   1                 1.0
1   2                 1.0
2   3                 0.0
3   4                 1.0
4   5                 0.0
5   6         